# Experiment: Tender Search CatBoost Ranker Demo

???? ??????? ?????????? ????????? ???????? ??? ??????? ????? ??????: `retrieval -> candidate set -> CatBoost reranking`.

??? ??????:
- ???????? `train / val / test` ???????
- ?????????? ranking features
- ????????? ? ??????? baseline ?? ??????? ??????? ?????????
- ???????? `CatBoostRanker`
- ?????? ?? `NDCG@K`, `HitRate@K`, `MRR@K`, `MAP@K`
- ?????? feature importance

## 1. ?????????? ?????????

????? ???????? ???????? ??????????? ?? `ML/requirements-notebook.txt`.

???? ? ??????? ??? ??? `item_title` ? `item_description`, ??????? ?????????? ?? ????????:

```bash
python ML/tools/build_ml_splits.py
```

In [ ]:
from pathlib import Path
import sys


def find_ml_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'tools').exists() and (candidate / 'data').exists():
            return candidate
        if (candidate / 'ML' / 'tools').exists() and (candidate / 'ML' / 'data').exists():
            return candidate / 'ML'
    raise RuntimeError('ML root not found')


ML_ROOT = find_ml_root(Path.cwd())
TOOLS_DIR = ML_ROOT / 'tools'
if str(TOOLS_DIR) not in sys.path:
    sys.path.append(str(TOOLS_DIR))

print(f'ML root: {ML_ROOT}')
print(f'Tools dir: {TOOLS_DIR}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from train_ranker import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_features,
    evaluate_ranking,
    feature_importance_frame,
    load_split_frames,
    position_baseline_scores,
    predict_scores,
    train_catboost_ranker,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

## 2. ???????? train / val / test

?????????? ??? ?????????????? ranking-?????? ?? synthetic dataset.

In [ ]:
splits_dir = ML_ROOT / 'data' / 'synthetic' / 'splits'
frames = load_split_frames(splits_dir)

pd.DataFrame(
    [
        {'split': name, 'rows': len(frame), 'sessions': frame['session_id'].nunique()}
        for name, frame in frames.items()
    ]
)

In [ ]:
preview_columns = [
    column
    for column in [
        'session_id',
        'query',
        'normalized_query',
        'ste_id',
        'position',
        'label',
        'item_title',
        'item_category_name',
        'item_supplier_name',
    ]
    if column in frames['train'].columns
]

frames['train'][preview_columns].head(10)

## 3. ?????????? ?????

`train_ranker.py` ?????? ????????, ??????? ?????? ????????? ?? ??????:
- ????? ? ??? ???????
- ?????????? ????????? ? ?????????? ? ???????? ????????????
- overlap ????? query ? title / description / attributes
- ???????? ???????????? ? ??????? ????????????
- ?????????????? ???????? ?? ??????

In [ ]:
train_df = build_features(frames['train'])
val_df = build_features(frames['val'])
test_df = build_features(frames['test'])

feature_preview = [
    'query',
    'normalized_query',
    'item_title',
    'item_category_name',
    'position_num',
    'cat_match_user',
    'supplier_match_user',
    'title_term_overlap',
    'description_term_overlap',
    'attribute_term_overlap',
    'total_term_overlap',
    'label',
]

train_df[feature_preview].head(10)

In [ ]:
pd.DataFrame({
    'numeric_features': pd.Series(NUMERIC_FEATURES),
    'categorical_features': pd.Series(CATEGORICAL_FEATURES + [''] * (len(NUMERIC_FEATURES) - len(CATEGORICAL_FEATURES)))
}).head(25)

## 4. Baseline ?? ??????? ?????????

??? ??????? ????????: ??? ???? ???????? ????? ? ???????? ??????, ??? ???? ??? baseline score.

In [ ]:
top_ks = (5, 10)
positive_threshold = 2

baseline_val_metrics = evaluate_ranking(
    val_df.assign(position_baseline=position_baseline_scores(val_df)),
    pred_col='position_baseline',
    top_ks=top_ks,
    positive_threshold=positive_threshold,
)
baseline_test_metrics = evaluate_ranking(
    test_df.assign(position_baseline=position_baseline_scores(test_df)),
    pred_col='position_baseline',
    top_ks=top_ks,
    positive_threshold=positive_threshold,
)

pd.DataFrame([baseline_val_metrics, baseline_test_metrics], index=['val', 'test'])

## 5. ???????? CatBoostRanker

????? ???????????? group-wise ranking objective. ?????? = ????????? ?????? (`session_id`).

In [ ]:
model, params = train_catboost_ranker(
    train_df,
    val_df,
    iterations=350,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
)

params

In [ ]:
val_pred = predict_scores(model, val_df)
test_pred = predict_scores(model, test_df)

catboost_val_metrics = evaluate_ranking(
    val_df.assign(prediction=val_pred),
    pred_col='prediction',
    top_ks=top_ks,
    positive_threshold=positive_threshold,
)
catboost_test_metrics = evaluate_ranking(
    test_df.assign(prediction=test_pred),
    pred_col='prediction',
    top_ks=top_ks,
    positive_threshold=positive_threshold,
)

comparison = pd.DataFrame(
    [baseline_val_metrics, catboost_val_metrics, baseline_test_metrics, catboost_test_metrics],
    index=['baseline_val', 'catboost_val', 'baseline_test', 'catboost_test'],
)
comparison

In [ ]:
uplift = pd.DataFrame({
    'val_uplift': pd.Series(catboost_val_metrics) - pd.Series(baseline_val_metrics),
    'test_uplift': pd.Series(catboost_test_metrics) - pd.Series(baseline_test_metrics),
})
uplift

## 6. Feature importance

??? ????? ???????? ??????? ??? ???????????: ??? ??????????, ????? ??????? ?????? ??????? ?????????? ??????? ?????.

In [ ]:
importance = feature_importance_frame(model, train_df)
importance.head(15)

In [ ]:
top_importance = importance.head(15).sort_values('importance')
ax = top_importance.plot.barh(x='feature', y='importance', figsize=(10, 6), legend=False, color='#2f6fed')
ax.set_title('Top 15 CatBoost features')
ax.set_xlabel('Importance')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 7. ??? ??? ???????? ?? ??????

??????? ???????? ???????? ????????????:

1. ????????, ??? retrieval ??? ???? candidate set.
2. ???????? baseline ?? ??????? ? `CatBoostRanker` ?? `NDCG@10`.
3. ???????? top feature importance ? ?????????, ??? ?????? ????????? ?? ?????? ?????, ?? ? ??????? ????????????.
4. ???????? ???????????, ??? ??? ????????????? ML backend ????? ?????????? ?? baseline ranking.

???? ????? ????????? ????????? ????? ????????????, ????????? CLI:

```bash
python ML/tools/train_ranker.py --artifacts-dir ML/models/catboost_ranker_v1
```